# RNN - Recurrent Neural Network

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [ ]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [ ]:
def tokenize(text):
  text = text.lower()
  text = text.replace("?", "")
  text = text.replace("'", "")
  return text.split()

In [ ]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [ ]:
vocab = {'<UNK>': 0}

In [ ]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_tokens = tokenized_question + tokenized_answer
  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [ ]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
len(vocab)

324

In [ ]:
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [ ]:
text_to_indices("what is campusx", vocab)

[1, 2, 0]

## Dataset and DataLoader

In [ ]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, idx):
    numericalized_question = text_to_indices(self.df.iloc[idx]['question'], self.vocab)
    numericalized_answer = text_to_indices(self.df.iloc[idx]['answer'], self.vocab)
    return torch.tensor(numericalized_question), torch.tensor(numericalized_answer)

In [ ]:
dataset = QADataset(df, vocab)

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

## Model

In [ ]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, x):
    x = self.embedding(x)
    _, final = self.rnn(x)
    x = self.fc(final.squeeze(0))
    return x

In [ ]:
learning_rate = 0.001
epochs = 20

In [ ]:
model = SimpleRNN(len(vocab))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    optimizer.zero_grad()
    output = model(question)
    loss = criterion(output, answer[0])
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_loss/len(dataloader)}")

Epoch: 1, Loss: 5.845021687613594
Epoch: 2, Loss: 5.091227446662055
Epoch: 3, Loss: 4.2020226637522375
Epoch: 4, Loss: 3.5327158053716023
Epoch: 5, Loss: 2.956993446085188
Epoch: 6, Loss: 2.42272154490153
Epoch: 7, Loss: 1.9377197603384653
Epoch: 8, Loss: 1.513260045978758
Epoch: 9, Loss: 1.1614207244581647
Epoch: 10, Loss: 0.8930383870999018
Epoch: 11, Loss: 0.6935274991724226
Epoch: 12, Loss: 0.5406159695651797
Epoch: 13, Loss: 0.4284263600905736
Epoch: 14, Loss: 0.3488022110528416
Epoch: 15, Loss: 0.2836864652733008
Epoch: 16, Loss: 0.2376961918340789
Epoch: 17, Loss: 0.1995600837800238
Epoch: 18, Loss: 0.1696577147477203
Epoch: 19, Loss: 0.1462169920404752
Epoch: 20, Loss: 0.12605636078450416


## Eval

In [ ]:
def predict(model, question, threshold=0.5):
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  output = model(question_tensor)
  probs = torch.nn.functional.softmax(output, dim=1)
  value, index = torch.max(probs, dim=1)
  if value < threshold:
    print("I don't know the answer to this question")
  else:
    print(list(vocab.keys())[index])

In [ ]:
predict(model, 'what is campusx')

I don't know the answer to this question


In [ ]:
predict(model, 'what is the largest planet in our solar system')

jupiter


# LSTM - Long Short Term Memory Neural Network

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [30]:
document = '''
I have wished a bird would fly away,
And not sing by my house all day;

Have clapped my hands at him from the door
When it seemed as if I could bear no more.

The fault must partly have been in me.
The bird was not to blame for his key.

And of course there must be something wrong
In wanting to silence any song.

He halted in the wind, and — what was that
Far in the maples, pale, but not a ghost?
He stood there bringing March against his thought,
And yet too ready to believe the most.

"Oh, that's the Paradise-in-bloom," I said;
And truly it was fair enough for flowers
had we but in us to assume in march
Such white luxuriance of May for ours.

We stood a moment so in a strange world,
Myself as one his own pretense deceives;
And then I said the truth (and we moved on).
A young beech clinging to its last year's leaves.

The farmhouse lingers, though averse to square
With the new city street it has to wear
A number in. But what about the brook
That held the house as in an elbow-crook?
I ask as one who knew the brook, its strength
And impulse, having dipped a finger length
And made it leap my knuckle, having tossed
A flower to try its currents where they crossed.
The meadow grass could be cemented down
From growing under pavements of a town;
The apple trees be sent to hearth-stone flame.
Is water wood to serve a brook the same?
How else dispose of an immortal force
No longer needed? Staunch it at its source
With cinder loads dumped down? The brook was thrown
Deep in a sewer dungeon under stone
In fetid darkness still to live and run —
And all for nothing it had ever done
Except forget to go in fear perhaps.
No one would know except for ancient maps
That such a brook ran water. But I wonder
If from its being kept forever under,
The thoughts may not have risen that so keep
This new-built city from both work and sleep.

MIST- I don't believe the sleepers in the house know where they are.
SMOKE- They've been here long enough to push the woods back from around the house and part them in the middle with a path.
MIST- And still I doubt if they know where they are. And I begin to fear they never will. All they maintain the path for is the comfort of visiting with the equally bewildered. Nearer in plight their neighbors are than distance.
SMOKE- I am the guardian wraith of starlit smoke that leans out this and that way from their chimney. I will not have their happiness despaired of.
MIST- No one - not I would give them up for lost simply because they don't know where they are.  I am the damper counterpart of smoke that gives off from a garden ground at night. But lifts no higher than a garden grows. I cotton to their landscape. That's who I am. I am no further from their fate than you are
SMOKE- They must by now have learned the native tongue. Why don't they ask the red man where they are?
MIST - They often do, and none the wiser for it. So do they also ask philosophers who come to look in on them from the pulpit. They will ask anyone there is to ask - In the fond faith accumulated fact - will of itself take fire
and light the world up. Learning has been a part of their
religion.
SMOKE - If the day ever comes when they know who they are, they may know better where they are. But who they are is too much to believe - either for them or the onlooking world. They are too sudden to be credible.
MIST - Listen, they murmur talking in the dark on what
should be their daylong theme continued. Putting the lamp out has not put their thought out. Let us pretend the dewdrops from the leaves are you, and I evesdropping on their unrest - a mist and smoke evesdropping on a haze - And see if we can tell the bass from the soprano.
THAN SMOKE AND MIST WHO BETTER COULD APPRAISE - THE KINDERED SPIRIT OF AN INNER HAZE!
SMOKE - If the day ever comes

There sandy seems the golden sky
And golden seems the sandy plain.
No habitation meets the eye
Unless in the horizon rim,
Some halfway up the limestone wall,
That spot of black is not a stain
Or shadow, but a cavern hole,
Where someone used to climb and crawl
To rest from his besetting fears.
I see the callus on his soul
The disappearing last of him
And of his race starvation slim,
Oh years ago - ten thousand years.

I had withdrawn in forest, and my song
Was swallowed up in leaves that blew alway;
And to the forest edge you came one day
(This was my dream) and looked and pondered long,
But did not enter, though the wish was strong:
You shook your pensive head as who should say,
‘I dare not—too far in his footsteps stray—
He must seek me would he undo the wrong.

Not far, but near, I stood and saw it all
Behind low boughs the trees let down outside;
And the sweet pang it cost me not to call
And tell you that I saw does still abide.
But 'tis not true that thus I dwelt aloof,
For the wood wakes, and you are here for proof.

Old Davis owned a solid mica mountain
In Dalton that would someday make his fortune.
There'd been some Boston people out to see it:
And experts said that deep down in the mountain
The mica sheets were big as plate-glass windows.
He'd like to take me there and show it to me.

'I'll tell you what you show me. You remember
You said you knew the place where once, on Kinsman,
The early Mormons made a settlement
And built a stone baptismal font outdoors-
But Smith, or someone, called them off the mountain
To go West to a worse fight with the desert.
You said you'd seen the stone baptismal font.
Well, take me there.'

Someday I will.'

'Today.'

'Huh, that old bathtub, what is that to see?
Let's talk about it.'

'Let's go see the place.'

'To shut you up I'll tell you what I'll do:
I'll find that fountain if it takes all summer,
And both of our united strengths, to do it.'

'You've lost it, then?'

'Not so but I can find it.
No doubt it's grown up some to woods around it.
The mountain may have shifted since I saw it
In eighty-five.'

'As long ago as that?'

'If I remember rightly, it had sprung
A leak and emptied then. And forty years
Can do a good deal to bad masonry.
You won't see any Mormon swimming in it.
But you have said it, and we're off to find it.
Old as I am, I'm going to let myself
Be dragged by you all over everywhere- '
'I thought you were a guide.'

'I am a guide,
And that's why I can't decently refuse you.'

We made a day of it out of the world,
Ascending to descend to reascend.
The old man seriously took his bearings,
And spoke his doubts in every open place.

We came out on a look-off where we faced
A cliff, and on the cliff a bottle painted,
Or stained by vegetation from above,
A likeness to surprise the thrilly tourist.

'Well, if I haven't brought you to the fountain,
At least I've brought you to the famous Bottle.'

'I won't accept the substitute. It's empty.'

'So's everything.'

'I want my fountain.'

'I guess you'd find the fountain just as empty.
And anyway this tells me where I am.'

'Hadn't you long suspected where you were?'

'You mean miles from that Mormon settlement?
Look here, you treat your guide with due respect
If you don't want to spend the night outdoors.
I vow we must be near the place from where
The two converging slides, the avalanches,
On Marshall, look like donkey's ears.
We may as well see that and save the day.'

'Don't donkey's ears suggest we shake our own?'

'For God's sake, aren't you fond of viewing nature?
You don't like nature. All you like is books.
What signify a donkey's cars and bottle,
However natural? Give you your books!
Well then, right here is where I show you books.
Come straight down off this mountain just as fast
As we can fall and keep a-bouncing on our feet.
It's hell for knees unless done hell-for-leather.'

Be ready, I thought, for almost anything.

We struck a road I didn't recognize,
But welcomed for the chance to lave my shoes
In dust once more. We followed this a mile,
Perhaps, to where it ended at a house
I didn't know was there. It was the kind
To bring me to for broad-board paneling.
I never saw so good a house deserted.

'Excuse me if I ask you in a window
That happens to be broken, Davis said.
'The outside doors as yet have held against us.
I want to introduce you to the people
Who used to live here. They were Robinsons.
You must have heard of Clara Robinson,
The poetess who wrote the book of verses
And had it published. It was all about
The posies on her inner windowsill,
And the birds on her outer windowsill,
And how she tended both, or had them tended:
She never tended anything herself.
She was 'shut in' for life. She lived her whole
Life long in bed, and wrote her things in bed.
I'll show You how she had her sills extended
To entertain the birds and hold the flowers.
Our business first's up attic with her books.'

We trod uncomfortably on crunching glass
Through a house stripped of everything
Except, it seemed, the poetess's poems.
Books, I should say!- if books are what is needed.
A whole edition in a packing case
That, overflowing like a horn of plenty,
Or like the poetess's heart of love,
Had spilled them near the window, toward the light
Where driven rain had wet and swollen them.
Enough to stock a village library-
Unfortunately all of one kind, though.
They bad been brought home from some publisher
And taken thus into the family.
Boys and bad hunters had known what to do
With stone and lead to unprotected glass:
Shatter it inward on the unswept floors.
How had the tender verse escaped their outrage?
By being invisible for what it was,
Or else by some remoteness that defied them
To find out what to do to hurt a poem.
Yet oh! the tempting flatness of a book,
To send it sailing out the attic window
Till it caught wind and, opening out its covers,
Tried to improve on sailing like a tile
By flying like a bird (silent in flight,
But all the burden of its body song),
Only to tumble like a stricken bird,
And lie in stones and bushes unretrieved.
Books were not thrown irreverently about.
They simply lay where someone now and then,
Having tried one, had dropped it at his feet
And left it lying where it fell rejected.
Here were all those the poetess's life
Had been too short to sell or give away.

'Take one,' Old Davis bade me graciously.

'Why not take two or three?'

'Take all you want.'
Good-looking books like that.' He picked one fresh
In virgin wrapper from deep in the box,
And stroked it with a horny-handed kindness.
He read in one and I read in another,
Both either looking for or finding something.

The attic wasps went missing by like bullets.

I was soon satisfied for the time being.

All the way home I kept remembering
The small book in my pocket. It was there.
The poetess had sighed, I knew, in heaven
At having eased her heart of one more copy-
Legitimately. My demand upon her,
Though slight, was a demand. She felt the tug.
In time she would be rid of all her books.

A neighbor of mine in the village
Likes to tell how one spring
When she was a girl on the farm, she did
A childlike thing.

One day she asked her father
To give her a garden plot
To plant and tend and reap herself,
And he said, 'Why not?'

In casting about for a corner
He thought of an idle bit
Of walled-off ground where a shop had stood,
And he said, 'Just it.'

And he said, 'That ought to make you
An ideal one-girl farm,
And give you a chance to put some strength
On your slim-jim arm.'

It was not enough of a garden
Her father said, to plow;
So she had to work it all by hand,
But she don't mind now.

She wheeled the dung in a wheelbarrow
Along a stretch of road;
But she always ran away and left
Her not-nice load,

And hid from anyone passing.
And then she begged the seed.
She says she thinks she planted one
Of all things but weed.

A hill each of potatoes,
Radishes, lettuce, peas,
Tomatoes, beets, beans, pumpkins, corn,
And even fruit trees.

And yes, she has long mistrusted
That a cider-apple
In bearing there today is hers,
Or at least may be.

Her crop was a miscellany
When all was said and done,
A little bit of everything,
A great deal of none.

Now when she sees in the village
How village things go,
Just when it seems to come in right,
She says, 'I know!

'It's as when I was a farmer...'
Oh never by way of advice!
And she never sins by telling the tale
To the same person twice.

To think to know the country and now know
The hillside on the day the sun lets go
Ten million silver lizards out of snow!
As often as I've seen it done before
I can't pretend to tell the way it's done.
It looks as if some magic of the sun
Lifted the rug that bred them on the floor
And the light breaking on them made them run.
But if I though to stop the wet stampede,
And caught one silver lizard by the tail,
And put my foot on one without avail,
And threw myself wet-elbowed and wet-kneed
In front of twenty others' wriggling speed,-
In the confusion of them all aglitter,
And birds that joined in the excited fun
By doubling and redoubling song and twitter,
I have no doubt I'd end by holding none.

It takes the moon for this. The sun's a wizard
By all I tell; but so's the moon a witch.
From the high west she makes a gentle cast
And suddenly, without a jerk or twitch,
She has her speel on every single lizard.
I fancied when I looked at six o'clock
The swarm still ran and scuttled just as fast.
The moon was waiting for her chill effect.
I looked at nine: the swarm was turned to rock
In every lifelike posture of the swarm,
Transfixed on mountain slopes almost erect.
Across each other and side by side they lay.
The spell that so could hold them as they were
Was wrought through trees without a breath of storm
To make a leaf, if there had been one, stir.
One lizard at the end of every ray.
The thought of my attempting such a stray!

When I go up through the mowing field,
The headless aftermath,
   Smooth-laid like thatch with the heavy dew,
 Half closes the garden path.

   And when I come to the garden ground,
The whir of sober birds
 Up from the tangle of withered weeds
 Is sadder than any words

A tree beside the wall stands bare,
 But a leaf that lingered brown,
   Disturbed, I doubt not, by my thought,
 Comes softly rattling down.

 I end not far from my going forth
   By picking the faded blue
Of the last remaining aster flower
 To carry again to you.

The line-storm clouds fly tattered and swift.
 The road is forlorn all day,
Where a myriad snowy quartz stones lift,
 And the hoof-prints vanish away.
The roadside flowers, too wet for the bee,
 Expend their bloom in vain.
Come over the hills and far with me,
 And be my love in the rain.

The birds have less to say for themselves
 In the wood-world's torn despair
Than now these numberless years the elves,
 Although they are no less there:
All song of the woods is crushed like some
 Wild, earily shattered rose.
Come, be my love in the wet woods, come,
 Where the boughs rain when it blows.

There is the gale to urge behind
 And bruit our singing down,
And the shallow waters aflutter with wind
 From which to gather your gown.
What matter if we go clear to the west,
 And come not through dry-shod?
For wilding brooch shall wet your breast
 The rain-fresh goldenrod.

Oh, never this whelming east wind swells
 But it seems like the sea's return
To the ancient lands where it left the shells
 Before the age of the fern;
And it seems like the time when after doubt
 Our love came back amain.
Oh, come forth into the storm and rout
 And be my love in the rain.

To Ridgely Torrence
On Last Looking into His 'Hesperides'

I often see flowers from a passing car
That are gone before I can tell what they are.

I want to get out of the train and go back
To see what they were beside the track.

I name all the flowers I am sure they weren't;
Not fireweed loving where woods have burnt-

Not bluebells gracing a tunnel mouth-
Not lupine living on sand and drouth.

Was something brushed across my mind
That no one on earth will ever find?

Heaven gives it glimpses only to those
Not in position to look too close.

There's a patch of old snow in a corner
That I should have guessed
Was a blow-away paper the rain
Had brought to rest.

It is speckled with grime as if
Small print overspread it,
The news of a day I've forgotten —
If I ever read it.

Dust always blowing about the town,
Except when sea-fog laid it down,
And I was one of the children told
Some of the blowing dust was gold.

All the dust the wind blew high
Appeared like god in the sunset sky,
But I was one of the children told
Some of the dust was really gold.

Such was life in the Golden Gate:
Gold dusted all we drank and ate,
And I was one of the children told,
'We all must eat our peck of gold.'

Oh, give us pleasure in the flowers today;
And give us not to think so far away
As the uncertain harvest; keep us here
All simply in the springing of the year.

Oh, give us pleasure in the orchard white,
Like nothing else by day, like ghosts by night;
And make us happy in the happy bees,
The swarm dilating round the perfect trees.

And make us happy in the darting bird
That suddenly above the bees is heard,
The meteor that thrusts in with needle bill,
And off a blossom in mid air stands still.

For this is love and nothing else is love,
To which it is reserved for God above
To sanctify to what far ends he will,
But which it only needs that we fulfill.

A voice said, Look me in the stars
And tell me truly, men of earth,
If all the soul-and-body scars
Were not too much to pay for birth.

I didn't make you know how glad I was
To have you come and camp here on our land.
I promised myself to get down some day
And see the way you lived, but I don't know!
With a houseful of hungry men to feed
I guess you'd find…. It seems to me
I can't express my feelings any more
Than I can raise my voice or want to lift
My hand (oh, I can lift it when I have to).
Did ever you feel so? I hope you never.
It's got so I don't even know for sure
Whether I am glad, sorry, or anything.
There's nothing but a voice-like left inside
That seems to tell me how I ought to feel,
And would feel if I wasn't all gone wrong.
You take the lake. I look and look at it.
I see it's a fair, pretty sheet of water.
I stand and make myself repeat out loud
The advantages it has, so long and narrow,
Like a deep piece of some old running river
Cut short off at both ends. It lies five miles
Straight away through the mountain notch
From the sink window where I wash the plates,
And all our storms come up toward the house,
Drawing the slow waves whiter and whiter and whiter.
It took my mind off doughnuts and soda biscuit
To step outdoors and take the water dazzle
A sunny morning, or take the rising wind
About my face and body and through my wrapper,
When a storm threatened from the Dragon's Den,
And a cold chill shivered across the lake.
I see it's a fair, pretty sheet of water,
Our Willoughby! How did you hear of it?
I expect, though, everyone's heard of it.
In a book about ferns? Listen to that!
You let things more like feathers regulate
Your going and coming. And you like it here?
I can see how you might. But I don't know!
It would be different if more people came,
For then there would be business. As it is,
The cottages Len built, sometimes we rent them,
Sometimes we don't. We've a good piece of shore
That ought to be worth something, and may yet.
But I don't count on it as much as Len.
He looks on the bright side of everything,
Including me. He thinks I'll be all right
With doctoring. But it's not medicine—
Lowe is the only doctor's dared to say so—
It's rest I want—there, I have said it out—
From cooking meals for hungry hired men
And washing dishes after them—from doing
Things over and over that just won't stay done.
By good rights I ought not to have so much
Put on me, but there seems no other way.
Len says one steady pull more ought to do it.
He says the best way out is always through.
And I agree to that, or in so far
As that I can see no way out but through—
Leastways for me—and then they'll be convinced.
It's not that Len don't want the best for me.
It was his plan our moving over in
Beside the lake from where that day I showed you
We used to live—ten miles from anywhere.
We didn't change without some sacrifice,
But Len went at it to make up the loss.
His work's a man's, of course, from sun to sun,
But he works when he works as hard as I do—
Though there's small profit in comparisons.
(Women and men will make them all the same.)
But work ain't all. Len undertakes too much.
He's into everything in town. This year
It's highways, and he's got too many men
Around him to look after that make waste.
They take advantage of him shamefully,
And proud, too, of themselves for doing so.
We have four here to board, great good-for-nothings,
Sprawling about the kitchen with their talk
While I fry their bacon. Much they care!
No more put out in what they do or say
Than if I wasn't in the room at all.
Coming and going all the time, they are:
I don't learn what their names are, let alone
Their characters, or whether they are safe
To have inside the house with doors unlocked.
I'm not afraid of them, though, if they're not
Afraid of me. There's two can play at that.
I have my fancies: it runs in the family.
My father's brother wasn't right. They kept him
Locked up for years back there at the old farm.
I've been away once—yes, I've been away.
The State Asylum. I was prejudiced;
I wouldn't have sent anyone of mine there;
You know the old idea—the only asylum
Was the poorhouse, and those who could afford,
Rather than send their folks to such a place,
Kept them at home; and it does seem more human.
But it's not so: the place is the asylum.
There they have every means proper to do with,
And you aren't darkening other people's lives—
Worse than no good to them, and they no good
To you in your condition; you can't know
Affection or the want of it in that state.
I've heard too much of the old-fashioned way.
My father's brother, he went mad quite young.
Some thought he had been bitten by a dog,
Because his violence took on the form
Of carrying his pillow in his teeth;
But it's more likely he was crossed in love,
Or so the story goes. It was some girl.
Anyway all he talked about was love.
They soon saw he would do someone a mischief
If he wa'n't kept strict watch of, and it ended
In father's building him a sort of cage,
Or room within a room, of hickory poles,
Like stanchions in the barn, from floor to ceiling,—
A narrow passage all the way around.
Anything they put in for furniture
He'd tear to pieces, even a bed to lie on.
So they made the place comfortable with straw,
Like a beast's stall, to ease their consciences.
Of course they had to feed him without dishes.
They tried to keep him clothed, but he paraded
With his clothes on his arm—all of his clothes.
Cruel—it sounds. I 'spose they did the best
They knew. And just when he was at the height,
Father and mother married, and mother came,
A bride, to help take care of such a creature,
And accommodate her young life to his.
That was what marrying father meant to her.
She had to lie and hear love things made dreadful
By his shouts in the night. He'd shout and shout
Until the strength was shouted out of him,
And his voice died down slowly from exhaustion.
He'd pull his bars apart like bow and bow-string,
And let them go and make them twang until
His hands had worn them smooth as any ox-bow.
And then he'd crow as if he thought that child's play—
The only fun he had. I've heard them say, though,
They found a way to put a stop to it.
He was before my time—I never saw him;
But the pen stayed exactly as it was
There in the upper chamber in the ell,
A sort of catch-all full of attic clutter.
I often think of the smooth hickory bars.
It got so I would say—you know, half fooling—
"It's time I took my turn upstairs in jail"—
Just as you will till it becomes a habit.
No wonder I was glad to get away.
Mind you, I waited till Len said the word.
I didn't want the blame if things went wrong.
I was glad though, no end, when we moved out,
And I looked to be happy, and I was,
As I said, for a while—but I don't know!
Somehow the change wore out like a prescription.
And there's more to it than just window-views
And living by a lake. I'm past such help—
Unless Len took the notion, which he won't,
And I won't ask him—it's not sure enough.
I 'spose I've got to go the road I'm going:
Other folks have to, and why shouldn't I?
I almost think if I could do like you,
Drop everything and live out on the ground—
But it might be, come night, I shouldn't like it,
Or a long rain. I should soon get enough,
And be glad of a good roof overhead.
I've lain awake thinking of you, I'll warrant,
More than you have yourself, some of these nights.
The wonder was the tents weren't snatched away
From over you as you lay in your beds.
I haven't courage for a risk like that.
Bless you, of course, you're keeping me from work,
But the thing of it is, I need to be kept.
There's work enough to do—there's always that;
But behind's behind. The worst that you can do
Is set me back a little more behind.
I sha'n't catch up in this world, anyway.
I'd rather you'd not go unless you must.

He is that fallen lance that lies as hurled,
That lies unlifted now, come dew, come rust,
But still lies pointed as it plowed the dust.
If we who sight along it round the world,
See nothing worthy to have been its mark,
It is because like men we look too near,
Forgetting that as fitted to the sphere,
Our missiles always make too short an arc.
They fall, they rip the grass, they intersect
The curve of earth, and striking, break their own;
They make us cringe for metal-point on stone.
But this we know, the obstacle that checked
And tripped the body, shot the spirit on
Further than target ever showed or shone.


Never tell me that not one star of all
That slip from heaven at night and softly fall
Has been picked up with stones to build a wall.

Some laborer found one faded and stone-cold,
And saving that its weight suggested gold
And tugged it from his first too certain hold,

He noticed nothing in it to remark.
He was not used to handling stars thrown dark
And lifeless from an interrupted arc.

He did not recognize in that smooth coal
The one thing palpable besides the soul
To penetrate the air in which we roll.

He did not see how like a flying thing
It brooded ant eggs, and bad one large wing,
One not so large for flying in a ring,

And a long Bird of Paradise's tail
(Though these when not in use to fly and trail
It drew back in its body like a snail):

Nor know that be might move it from the spot—
The harm was done: from having been star-shot
The very nature of the soil was hot

And burning to yield flowers instead of grain,
Flowers fanned and not put out by all the rain
Poured on them by his prayers prayed in vain.

He moved it roughly with an iron bar,
He loaded an old stoneboat with the star
And not, as you might think, a flying car,

Such as even poets would admit perforce
More practical than Pegasus the horse
If it could put a star back in its course.

He dragged it through the plowed ground at a pace
But faintly reminiscent of the race
Of jostling rock in interstellar space.

It went for building stone, and I, as though
Commanded in a dream, forever go
To right the wrong that this should have been so.

Yet ask where else it could have gone as well,
I do not know—I cannot stop to tell:
He might have left it lying where it fell.

From following walls I never lift my eye,
Except at night to places in the sky
Where showers of charted meteors let fly.

Some may know what they seek in school and church,
And why they seek it there; for what I search
I must go measuring stone walls, perch on perch;

Sure that though not a star of death and birth,
So not to be compared, perhaps, in worth
To such resorts of life as Mars and Earth—

Though not, I say, a star of death and sin,
It yet has poles, and only needs a spin
To show its worldly nature and begin

To chafe and shuffle in my calloused palm
And run off in strange tangents with my arm,
As fish do with the line in first alarm.

Such as it is, it promises the prize
Of the one world complete in any size
That I am like to compass, fool or wise.

When a friend calls to me from the road
And slows his horse to a meaning walk,
I don't stand still and look around
On all the hills I haven't hoed,
And shout from where I am, What is it?
No, not as there is a time to talk.
I thrust my hoe in the mellow ground,
Blade-end up and five feet tall,
And plod: I go up to the stone wall
For a friendly visit.

A winter garden in an alder swamp,
Where conies now come out to sun and romp,
As near a paradise as it can be
And not melt snow or start a dormant tree.

It lifts existence on a plane of snow
One level higher than the earth below,
One level nearer heaven overhead,
And last year's berries shining scarlet red.

It lifts a gaunt luxuriating beast
Where he can stretch and hold his highest feat
On some wild apple tree's young tender bark,
What well may prove the year's high girdle mark.

So near to paradise all pairing ends:
Here loveless birds now flock as winter friends,
Content with bud-inspecting. They presume
To say which buds are leaf and which are bloom.

A feather-hammer gives a double knock.
This Eden day is done at two o'clock.
An hour of winter day might seem too short
To make it worth life's while to wake and sport.

When the spent sun throws up its rays on cloud
And goes down burning into the gulf below,
No voice in nature is heard to cry aloud
At what has happened. Birds, at least must know
It is the change to darkness in the sky.
Murmuring something quiet in her breast,
One bird begins to close a faded eye;
Or overtaken too far from his nest,
Hurrying low above the grove, some waif
Swoops just in time to his remembered tree.
At most he thinks or twitters softly, 'Safe!
Now let the night be dark for all of me.
Let the night be too dark for me to see
Into the future. Let what will be, be.'

When I see birches bend to left and right
Across the lines of straighter darker trees,
I like to think some boy's been swinging them.
But swinging doesn't bend them down to stay.
Ice-storms do that. Often you must have seen them
Loaded with ice a sunny winter morning
After a rain. They click upon themselves
As the breeze rises, and turn many-colored
As the stir cracks and crazes their enamel.
Soon the sun's warmth makes them shed crystal shells
Shattering and avalanching on the snow-crust—
Such heaps of broken glass to sweep away
You'd think the inner dome of heaven had fallen.
They are dragged to the withered bracken by the load,
And they seem not to break; though once they are bowed
So low for long, they never right themselves:
You may see their trunks arching in the woods
Years afterwards, trailing their leaves on the ground
Like girls on hands and knees that throw their hair
Before them over their heads to dry in the sun.
But I was going to say when Truth broke in
With all her matter-of-fact about the ice-storm
(Now am I free to be poetical?)
I should prefer to have some boy bend them
As he went out and in to fetch the cows—
Some boy too far from town to learn baseball,
Whose only play was what he found himself,
Summer or winter, and could play alone.
One by one he subdued his father's trees
By riding them down over and over again
Until he took the stiffness out of them,
And not one but hung limp, not one was left
For him to conquer. He learned all there was
To learn about not launching out too soon
And so not carrying the tree away
Clear to the ground. He always kept his poise
To the top branches, climbing carefully
With the same pains you use to fill a cup
Up to the brim, and even above the brim.
Then he flung outward, feet first, with a swish,
Kicking his way down through the air to the ground.
So was I once myself a swinger of birches.
And so I dream of going back to be.
It's when I'm weary of considerations,
And life is too much like a pathless wood
Where your face burns and tickles with the cobwebs
Broken across it, and one eye is weeping
From a twig's having lashed across it open.
I'd like to get away from earth awhile
And then come back to it and begin over.
May no fate willfully misunderstand me
And half grant what I wish and snatch me away
Not to return. Earth's the right place for love:
I don't know where it's likely to go better.
I'd like to go by climbing a birch tree,
And climb black branches up a snow-white trunk
Toward heaven, till the tree could bear no more,
But dipped its top and set me down again.
That would be good both going and coming back.
One could do worse than be a swinger of birches.

I have been one acquainted with the night.
I have walked out in rain — and back in rain.
I have outwalked the furthest city light.

I have looked down the saddest city lane.
I have passed by the watchman on his beat
And dropped my eyes, unwilling to explain.

I have stood still and stopped the sound of feet
When far away an interrupted cry
Came over houses from another street,

But not to call me back or say good-bye;
And further still at an unearthly height,
A luminary clock against the sky

Proclaimed the time was neither wrong nor right.
I have been one acquainted with the night.

ONCE on the kind of day called "weather breeder,"
When the heat slowly hazes and the sun
By its own power seems to be undone,
I was half boring through, half climbing through
A swamp of cedar. Choked with oil of cedar
And scurf of plants, and weary and over-heated,
And sorry I ever left the road I knew,
I paused and rested on a sort of hook
That had me by the coat as good as seated,
And since there was no other way to look,
Looked up toward heaven, and there against the blue,
Stood over me a resurrected tree,
A tree that had been down and raised again—
A barkless spectre. He had halted too,
As if for fear of treading upon me.
I saw the strange position of his hands—
Up at his shoulders, dragging yellow strands
Of wire with something in it from men to men.
"You here?" I said. "Where aren't you nowadays
And what's the news you carry—if you know?
And tell me where you're off for—Montreal?
Me? I'm not off for anywhere at all.
Sometimes I wander out of beaten ways
Half looking for the orchid Calypso."

The city had withdrawn into itself
And left at last the country to the country;
When between whirls of snow not come to lie
And whirls of foliage not yet laid, there drove
A stranger to our yard, who looked the city,
Yet did in country fashion in that there
He sat and waited till he drew us out
A-buttoning coats to ask him who he was.
He proved to be the city come again
To look for something it had left behind
And could not do without and keep its Christmas.
He asked if I would sell my Christmas trees;
My woods—the young fir balsams like a place
Where houses all are churches and have spires.
I hadn’t thought of them as Christmas Trees.
I doubt if I was tempted for a moment
To sell them off their feet to go in cars
And leave the slope behind the house all bare,
Where the sun shines now no warmer than the moon.
I’d hate to have them know it if I was.
Yet more I’d hate to hold my trees except
As others hold theirs or refuse for them,
Beyond the time of profitable growth,
The trial by market everything must come to.
I dallied so much with the thought of selling.
Then whether from mistaken courtesy
And fear of seeming short of speech, or whether
From hope of hearing good of what was mine,
I said, “There aren’t enough to be worth while.”
“I could soon tell how many they would cut,
You let me look them over.”

“You could look.
But don’t expect I’m going to let you have them.”
Pasture they spring in, some in clumps too close
That lop each other of boughs, but not a few
Quite solitary and having equal boughs
All round and round. The latter he nodded “Yes” to,
Or paused to say beneath some lovelier one,
With a buyer’s moderation, “That would do.”
I thought so too, but wasn’t there to say so.
We climbed the pasture on the south, crossed over,
And came down on the north.
He said, “A thousand.”

“A thousand Christmas trees!—at what apiece?”

He felt some need of softening that to me:
“A thousand trees would come to thirty dollars.”

Then I was certain I had never meant
To let him have them. Never show surprise!
But thirty dollars seemed so small beside
The extent of pasture I should strip, three cents
(For that was all they figured out apiece),
Three cents so small beside the dollar friends
I should be writing to within the hour
Would pay in cities for good trees like those,
Regular vestry-trees whole Sunday Schools
Could hang enough on to pick off enough.
A thousand Christmas trees I didn’t know I had!
Worth three cents more to give away than sell,
As may be shown by a simple calculation.
Too bad I couldn’t lay one in a letter.
I can’t help wishing I could send you one,
In wishing you herewith a Merry Christmas.


Blueberries as big as the end of your thumb,
Real sky-blue, and heavy, and ready to drum
In the cavernous pail of the first one to come!
And all ripe together, not some of them green
And some of them ripe! You ought to have seen!

It is a blue-butterfly day here in spring,
And with these sky-flakes down in flurry on flurry
There is more unmixed color on the wing
Than flowers will show for days unless they hurry.

But these are flowers that fly and all but sing:
And now from having ridden out desire
They lie closed over in the wind and cling
Where wheels have freshly sliced the April mire.


I wandered lonely as a cloud
That floats on high o'er vales and hills,
When all at once I saw a crowd,
A host, of golden daffodils;
Beside the lake, beneath the trees,
Fluttering and dancing in the breeze.

Continuous as the stars that shine
And twinkle on the milky way,
They stretched in never-ending line
Along the margin of a bay:
Ten thousand saw I at a glance,
Tossing their heads in sprightly dance.

The waves beside them danced; but they
Out-did the sparkling waves in glee:
A poet could not but be gay,
In such a jocund company:
I gazed- and gazed- but little thought
What wealth the show to me had brought:

For oft, when on my couch I lie
In vacant or in pensive mood,
They flash upon that inward eye
Which is the bliss of solitude;
And then my heart with pleasure fills,
And dances with the daffodils.

I marvel how Nature could ever find space
For so many strange contrasts in one human face:
There's thought and no thought, and there's paleness and bloom
And bustle and sluggishness, pleasure and gloom.

There's weakness, and strength both redundant and vain;
Such strength as, if ever affliction and pain
Could pierce through a temper that's soft to disease,
Would be rational peace--a philosopher's ease.

There's indifference, alike when he fails or succeeds,
And attention full ten times as much as there needs;
Pride where there's no envy, there's so much of joy;
And mildness, and spirit both forward and coy.

There's freedom, and sometimes a diffident stare
Of shame scarcely seeming to know that she's there,
There's virtue, the title it surely may claim,
Yet wants heaven knows what to be worthy the name.

This picture from nature may seem to depart,
Yet the Man would at once run away with your heart;
And I for five centuries right gladly would be
Such an odd such a kind happy creature as he.

Lo! where the Moon along the sky
Sails with her happy destiny;
Oft is she hid from mortal eye
Or dimly seen,
But when the clouds asunder fly
How bright her mien!

Far different we--a froward race,
Thousands though rich in Fortune's grace
With cherished sullenness of pace
Their way pursue,
Ingrates who wear a smileless face
The whole year through.

If kindred humours e'er would make
My spirit droop for drooping's sake,
From Fancy following in thy wake,
Bright ship of heaven!
A counter impulse let me take
And be forgiven.

The world is too much with us; late and soon,
Getting and spending, we lay waste our powers:
Little we see in Nature that is ours;
We have given our hearts away, a sordid boon!
This Sea that bares her bosom to the moon;
The winds that will be howling at all hours,
And are up-gathered now like sleeping flowers;
For this, for everything, we are out of tune,
It moves us not.--Great God! I'd rather be
A Pagan suckled in a creed outworn;
So might I, standing on this pleasant lea,
Have glimpses that would make me less forlorn;
Have sight of Proteus rising from the sea;
Or hear old Triton blow his wreathed horn.

She dwelt among the untrodden ways
Beside the springs of Dove,
Maid whom there were none to praise
And very few to love:

A violet by a mossy stone
Half hidden from the eye!
---Fair as a star, when only one
Is shining in the sky.

She lived unknown, and few could know
When Lucy ceased to be;
But she is in her grave, and, oh,
The difference to me!

Calm is all nature as a resting wheel.
The kine are couched upon the dewy grass;
The horse alone, seen dimly as I pass,
Is cropping audibly his later meal:
Dark is the ground; a slumber seems to steal
O'er vale, and mountain, and the starless sky.
Now, in this blank of things, a harmony,
Home-felt, and home-created, comes to heal
That grief for which the senses still supply
Fresh food; for only then, when memory
Is hushed, am I at rest. My Friends! restrain
Those busy cares that would allay my pain;
Oh! leave me to myself, nor let me feel
The officious touch that makes me droop again.

Behold her, single in the field,
Yon solitary Highland Lass!
Reaping and singing by herself;
Stop here, or gently pass!
Alone she cuts and binds the grain,
And sings a melancholy strain;
O listen! for the Vale profound
Is overflowing with the sound.

No Nightingale did ever chaunt
More welcome notes to weary bands
Of travellers in some shady haunt,
Among Arabian sands:
A voice so thrilling ne'er was heard
In spring-time from the Cuckoo-bird,
Breaking the silence of the seas
Among the farthest Hebrides.

Will no one tell me what she sings?--
Perhaps the plaintive numbers flow
For old, unhappy, far-off things,
And battles long ago:
Or is it some more humble lay,
Familiar matter of to-day?
Some natural sorrow, loss, or pain,
That has been, and may be again?

Whate'er the theme, the Maiden sang
As if her song could have no ending;
I saw her singing at her work,
And o'er the sickle bending;--
I listened, motionless and still;
And, as I mounted up the hill,
The music in my heart I bore,
Long after it was heard no more.

There is a change- and I am poor;
Your love hath been, nor long ago,
A fountain at my fond heart's door,
Whose only business was to flow;
And flow it did; not taking heed
Of its own bounty, or my need.

What happy moments did I count!
Blest was I then all bliss above!
Now, for that consecrated fount
Of murmuring, sparkling, living love,
What have I? Shall I dare to tell?
A comfortless and hidden well.

A well of love- it may be deep-
I trust it is,- and never dry:
What matter? If the waters sleep
In silence and obscurity.
- Such change, and at the very door
Of my fond heart, hath made me poor.

------The sky is overcast
With a continuous cloud of texture close,
Heavy and wan, all whitened by the Moon,
Which through that veil is indistinctly seen,
A dull, contracted circle, yielding light
So feebly spread, that not a shadow falls,
Chequering the ground--from rock, plant, tree, or tower.
At length a pleasant instantaneous gleam
Startles the pensive traveller while he treads
His lonesome path, with unobserving eye
Bent earthwards; he looks up--the clouds are split
Asunder,--and above his head he sees
The clear Moon, and the glory of the heavens.
There, in a black-blue vault she sails along,
Followed by multitudes of stars, that, small
And sharp, and bright, along the dark abyss
Drive as she drives: how fast they wheel away,
Yet vanish not!--the wind is in the tree,
But they are silent;--still they roll along
Immeasurably distant; and the vault,
Built round by those white clouds, enormous clouds,
Still deepens its unfathomable depth.
At length the Vision closes; and the mind,
Not undisturbed by the delight it feels,
Which slowly settles into peaceful calm,
Is left to muse upon the solemn scene.

It was an April morning: fresh and clear
The Rivulet, delighting in its strength,
Ran with a young man's speed; and yet the voice
Of waters which the winter had supplied
Was softened down into a vernal tone.
The spirit of enjoyment and desire,
And hopes and wishes, from all living things
Went circling, like a multitude of sounds.
The budding groves seemed eager to urge on
The steps of June; as if their various hues
Were only hindrances that stood between
Them and their object: but, meanwhile, prevailed
Such an entire contentment in the air
That every naked ash, and tardy tree
Yet leafless, showed as if the countenance
With which it looked on this delightful day
Were native to the summer.--Up the brook
I roamed in the confusion of my heart,
Alive to all things and forgetting all.
At length I to a sudden turning came
In this continuous glen, where down a rock
The Stream, so ardent in its course before,
Sent forth such sallies of glad sound, that all
Which I till then had heard, appeared the voice
Of common pleasure: beast and bird, the lamb,
The shepherd's dog, the linnet and the thrush
Vied with this waterfall, and made a song,
Which, while I listened, seemed like the wild growth
Or like some natural produce of the air,
That could not cease to be. Green leaves were here;
But 'twas the foliage of the rocks--the birch,
The yew, the holly, and the bright green thorn,
With hanging islands of resplendent furze:
And, on a summit, distant a short space,
By any who should look beyond the dell,
A single mountain-cottage might be seen.
I gazed and gazed, and to myself I said,
'Our thoughts at least are ours; and this wild nook,
My EMMA, I will dedicate to thee.'
----Soon did the spot become my other home,
My dwelling, and my out-of-doors abode.
And, of the Shepherds who have seen me there,
To whom I sometimes in our idle talk
Have told this fancy, two or three, perhaps,
Years after we are gone and in our graves,
When they have cause to speak of this wild place,
May call it by the name of EMMA'S DELL.

A narrow girdle of rough stones and crags,
A rude and natural causeway, interposed
Between the water and a winding slope
Of copse and thicket, leaves the eastern shore
Of Grasmere safe in its own privacy:
And there myself and two beloved Friends,
One calm September morning, ere the mist
Had altogether yielded to the sun,
Sauntered on this retired and difficult way.
----Ill suits the road with one in haste; but we
Played with our time; and, as we strolled along,
It was our occupation to observe
Such objects as the waves had tossed ashore--
Feather, or leaf, or weed, or withered bough,
Each on the other heaped, along the line
Of the dry wreck. And, in our vacant mood,
Not seldom did we stop to watch some tuft
Of dandelion seed or thistle's beard,
That skimmed the surface of the dead calm lake,
Suddenly halting now--a lifeless stand!
And starting off again with freak as sudden;
In all its sportive wanderings, all the while,
Making report of an invisible breeze
That was its wings, its chariot, and its horse,
Its playmate, rather say, its moving soul.
--And often, trifling with a privilege
Alike indulged to all, we paused, one now,
And now the other, to point out, perchance
To pluck, some flower or water-weed, too fair
Either to be divided from the place
On which it grew, or to be left alone
To its own beauty. Many such there are,
Fair ferns and flowers, and chiefly that tall fern,
So stately, of the queen Osmunda named;
Plant lovelier, in its own retired abode
On Grasmere's beach, than Naiad by the side
Of Grecian brook, or Lady of the Mere,
Sole-sitting by the shores of old romance.
--So fared we that bright morning: from the fields
Meanwhile, a noise was heard, the busy mirth
Of reapers, men and women, boys and girls.
Delighted much to listen to those sounds,
And feeding thus our fancies, we advanced
Along the indented shore; when suddenly,
Through a thin veil of glittering haze was seen
Before us, on a point of jutting land,
The tall and upright figure of a Man
Attired in peasant's garb, who stood alone,
Angling beside the margin of the lake.
'Improvident and reckless,' we exclaimed,
'The Man must be, who thus can lose a day
Of the mid harvest, when the labourer's hire
Is ample, and some little might be stored
Wherewith to cheer him in the winter time.'
Thus talking of that Peasant, we approached
Close to the spot where with his rod and line
He stood alone; whereat he turned his head
To greet us--and we saw a Mam worn down
By sickness, gaunt and lean, with sunken cheeks
And wasted limbs, his legs so long and lean
That for my single self I looked at them,
Forgetful of the body they sustained.--
Too weak to labour in the harvest field,
The Man was using his best skill to gain
A pittance from the dead unfeeling lake
That knew not of his wants. I will not say
What thoughts immediately were ours, nor how
The happy idleness of that sweet morn,
With all its lovely images, was changed
To serious musing and to self-reproach.
Nor did we fail to see within ourselves
What need there is to be reserved in speech,
And temper all our thoughts with charity.
--Therefore, unwilling to forget that day,
My Friend, Myself, and She who then received
The same admonishment, have called the place
By a memorial name, uncouth indeed
As e'er by mariner was given to bay
Or foreland, on a new-discovered coast;
And POINT RASH-JUDGMENT is the name it bears.

Five years have past; five summers, with the length
Of five long winters! and again I hear
These waters, rolling from their mountain-springs
With a soft inland murmur.--Once again
Do I behold these steep and lofty cliffs,
That on a wild secluded scene impress
Thoughts of more deep seclusion; and connect
The landscape with the quiet of the sky.
The day is come when I again repose
Here, under this dark sycamore, and view
These plots of cottage-ground, these orchard-tufts,
Which at this season, with their unripe fruits,
Are clad in one green hue, and lose themselves
'Mid groves and copses. Once again I see
These hedge-rows, hardly hedge-rows, little lines
Of sportive wood run wild: these pastoral farms,
Green to the very door; and wreaths of smoke
Sent up, in silence, from among the trees!
With some uncertain notice, as might seem
Of vagrant dwellers in the houseless woods,
Or of some Hermit's cave, where by his fire
The Hermit sits alone.
These beauteous forms,
Through a long absence, have not been to me
As is a landscape to a blind man's eye:
But oft, in lonely rooms, and 'mid the din
Of towns and cities, I have owed to them
In hours of weariness, sensations sweet,
Felt in the blood, and felt along the heart;
And passing even into my purer mind,
With tranquil restoration:--feelings too
Of unremembered pleasure: such, perhaps,
As have no slight or trivial influence
On that best portion of a good man's life,
His little, nameless, unremembered, acts
Of kindness and of love. Nor less, I trust,
To them I may have owed another gift,
Of aspect more sublime; that blessed mood,
In which the burthen of the mystery,
In which the heavy and the weary weight
Of all this unintelligible world,
Is lightened:--that serene and blessed mood,
In which the affections gently lead us on,--
Until, the breath of this corporeal frame
And even the motion of our human blood
Almost suspended, we are laid asleep
In body, and become a living soul:
While with an eye made quiet by the power
Of harmony, and the deep power of joy,
We see into the life of things.
If this
Be but a vain belief, yet, oh! how oft--
In darkness and amid the many shapes
Of joyless daylight; when the fretful stir
Unprofitable, and the fever of the world,
Have hung upon the beatings of my heart--
How oft, in spirit, have I turned to thee,
O sylvan Wye! thou wanderer thro' the woods,
How often has my spirit turned to thee!
And now, with gleams of half-extinguished thought,
With many recognitions dim and faint,
And somewhat of a sad perplexity,
The picture of the mind revives again:
While here I stand, not only with the sense
Of present pleasure, but with pleasing thoughts
That in this moment there is life and food
For future years. And so I dare to hope,
Though changed, no doubt, from what I was when first
I came among these hills; when like a roe
I bounded o'er the mountains, by the sides
Of the deep rivers, and the lonely streams,
Wherever nature led: more like a man
Flying from something that he dreads, than one
Who sought the thing he loved. For nature then
(The coarser pleasures of my boyish days,
And their glad animal movements all gone by)
To me was all in all.--I cannot paint
What then I was. The sounding cataract
Haunted me like a passion: the tall rock,
The mountain, and the deep and gloomy wood,
Their colours and their forms, were then to me
An appetite; a feeling and a love,
That had no need of a remoter charm,
By thought supplied, nor any interest
Unborrowed from the eye.--That time is past,
And all its aching joys are now no more,
And all its dizzy raptures. Not for this
Faint I, nor mourn nor murmur, other gifts
Have followed; for such loss, I would believe,
Abundant recompence. For I have learned
To look on nature, not as in the hour
Of thoughtless youth; but hearing oftentimes
The still, sad music of humanity,
Nor harsh nor grating, though of ample power
To chasten and subdue. And I have felt
A presence that disturbs me with the joy
Of elevated thoughts; a sense sublime
Of something far more deeply interfused,
Whose dwelling is the light of setting suns,
And the round ocean and the living air,
And the blue sky, and in the mind of man;
A motion and a spirit, that impels 0
All thinking things, all objects of all thought,
And rolls through all things. Therefore am I still
A lover of the meadows and the woods,
And mountains; and of all that we behold
From this green earth; of all the mighty world
Of eye, and ear,--both what they half create,
And what perceive; well pleased to recognise
In nature and the language of the sense,
The anchor of my purest thoughts, the nurse,
The guide, the guardian of my heart, and soul
Of all my moral being.
Nor perchance,
If I were not thus taught, should I the more
Suffer my genial spirits to decay:
For thou art with me here upon the banks
Of this fair river; thou my dearest Friend,
My dear, dear Friend; and in thy voice I catch
The language of my former heart, and read
My former pleasures in the shooting lights
Of thy wild eyes. Oh! yet a little while
May I behold in thee what I was once,
My dear, dear Sister! and this prayer I make,
Knowing that Nature never did betray
The heart that loved her; 'tis her privilege,
Through all the years of this our life, to lead
From joy to joy: for she can so inform
The mind that is within us, so impress
With quietness and beauty, and so feed
With lofty thoughts, that neither evil tongues,
Rash judgments, nor the sneers of selfish men,
Nor greetings where no kindness is, nor all
The dreary intercourse of daily life,
Shall e'er prevail against us, or disturb
Our cheerful faith, that all which we behold
Is full of blessings. Therefore let the moon
Shine on thee in thy solitary walk;
And let the misty mountain-winds be free
To blow against thee: and, in after years,
When these wild ecstasies shall be matured
Into a sober pleasure; when thy mind
Shall be a mansion for all lovely forms,
Thy memory be as a dwelling-place
For all sweet sounds and harmonies; oh! then,
If solitude, or fear, or pain, or grief,
Should be thy portion, with what healing thoughts
Of tender joy wilt thou remember me,
And these my exhortations! Nor, perchance--
If I should be where I no more can hear
Thy voice, nor catch from thy wild eyes these gleams
Of past existence--wilt thou then forget
That on the banks of this delightful stream
We stood together; and that I, so long
A worshipper of Nature, hither came
Unwearied in that service: rather say
With warmer love--oh! with far deeper zeal
Of holier love. Nor wilt thou then forget,
That after many wanderings, many years
Of absence, these steep woods and lofty cliffs,
And this green pastoral landscape, were to me
More dear, both for themselves and for thy sake!

A Whirl-Blast from behind the hill
Rushed o'er the wood with startling sound;
Then--all at once the air was still,
And showers of hailstones pattered round.
Where leafless oaks towered high above,
I sat within an undergrove
Of tallest hollies, tall and green;
A fairer bower was never seen.
From year to year the spacious floor
With withered leaves is covered o'er,
And all the year the bower is green.
But see! where'er the hailstones drop
The withered leaves all skip and hop;
There's not a breeze--no breath of air--
Yet here, and there, and everywhere
Along the floor, beneath the shade
By those embowering hollies made,
The leaves in myriads jump and spring,
As if with pipes and music rare
Some Robin Good-fellow were there,
And all those leaves, in festive glee,
Were dancing to the minstrelsy.

Strange fits of passion have I known:
And I will dare to tell,
But in the lover's ear alone,
What once to me befell.

When she I loved looked every day
Fresh as a rose in June,
I to her cottage bent my way,
Beneath an evening-moon.

Upon the moon I fixed my eye,
All over the wide lea;
With quickening pace my horse drew nigh
Those paths so dear to me.

And now we reached the orchard-plot;
And, as we climbed the hill,
The sinking moon to Lucy's cot
Came near, and nearer still.

In one of those sweet dreams I slept,
Kind Nature's gentlest boon!
And all the while my eye I kept
On the descending moon.

My horse moved on; hoof after hoof
He raised, and never stopped:
When down behind the cottage roof,
At once, the bright moon dropped.

What fond and wayward thoughts will slide
Into a Lover's head!
'O mercy!' to myself I cried,
'If Lucy hould be dead!'

A slumber did my spirit seal
I had no human fears:
She seemed a thing that could not feel
The touch of earthly years.

No motion has she now, no force;
She neither hears nor sees;
Rolled round in earth's diurnal course,
With rocks, and stones, and trees.

Earth has not anything to show more fair:
Dull would he be of soul who could pass by
A sight so touching in its majesty:
This City now doth, like a garment, wear
The beauty of the morning; silent, bare,
Ships, towers, domes, theatres, and temples lie
Open unto the fields, and to the sky;
All bright and glittering in the smokeless air.
Never did sun more beautifully steep
In his first splendour, valley, rock, or hill;
Ne'er saw I, never felt, a calm so deep!
The river glideth at his own sweet will:
Dear God! the very houses seem asleep;
And all that mighty heart is lying still!

My heart leaps up when I behold
A rainbow in the sky:
So was it when my life began;
So is it now I am a man;
So be it when I shall grow old,
Or let me die!
The Child is father of the Man;
And I could wish my days to be
Bound each to each by natural piety.

FAREWELL, thou little Nook of mountain-ground,
Thou rocky corner in the lowest stair
Of that magnificent temple which doth bound
One side of our whole vale with grandeur rare;
Sweet garden-orchard, eminently fair,
The loveliest spot that man hath ever found,
Farewell!--we leave thee to Heaven's peaceful care,
Thee, and the Cottage which thou dost surround.

Our boat is safely anchored by the shore,
And there will safely ride when we are gone;
The flowering shrubs that deck our humble door
Will prosper, though untended and alone:
Fields, goods, and far-off chattels we have none:
These narrow bounds contain our private store
Of things earth makes, and sun doth shine upon;
Here are they in our sight--we have no more.

Sunshine and shower be with you, bud and bell!
For two months now in vain we shall be sought:
We leave you here in solitude to dwell
With these our latest gifts of tender thought;
Thou, like the morning, in thy saffron coat,
Bright gowan, and marsh-marigold, farewell!
Whom from the borders of the Lake we brought,
And placed together near our rocky Well.

We go for One to whom ye will be dear;
And she will prize this Bower, this Indian shed,
Our own contrivance, Building without peer!
--A gentle Maid, whose heart is lowly bred,
Whose pleasures are in wild fields gathered,
With joyousness, and with a thoughtful cheer,
Will come to you; to you herself will wed;
And love the blessed life that we lead here.

Dear Spot! which we have watched with tender heed,
Bringing thee chosen plants and blossoms blown
Among the distant mountains, flower and weed,
Which thou hast taken to thee as thy own,
Making all kindness registered and known;
Thou for our sakes, though Nature's child indeed,
Fair in thyself and beautiful alone,
Hast taken gifts which thou dost little need.

And O most constant, yet most fickle Place,
Thou hast thy wayward moods, as thou dost show
To them who look not daily on thy face;
Who, being loved, in love no bounds dost know,
And say'st, when we forsake thee, 'Let them go!'
Thou easy-hearted Thing, with thy wild race
Of weeds and flowers, till we return be slow,
And travel with the year at a soft pace.

Help us to tell Her tales of years gone by,
And this sweet spring, the best beloved and best;
Joy will be flown in its mortality;
Something must stay to tell us of the rest.
Here, thronged with primroses, the steep rock's breast
Glittered at evening like a starry sky;
And in this bush our sparrow built her nest,
Of which I sang one song that will not die.

O happy Garden! whose seclusion deep
Hath been so friendly to industrious hours;
And to soft slumbers, that did gently steep
Our spirits, carrying with them dreams of flowers,
And wild notes warbled among leafy bowers;
Two burning months let summer overleap,
And, coming back with Her who will be ours,
Into thy bosom we again shall creep.

The Child is father of the Man;
And I could wish my days to be
Bound each to each by natural piety.


I

There was a time when meadow, grove, and stream,
The earth, and every common sight,
To me did seem
Apparelled in celestial light,
The glory and the freshness of a dream.
It is not now as it hath been of yore; -
Turn wheresoe'er I may,
By night or day,
The things which I have seen I now can see no more.


II

The Rainbow comes and goes,
And lovely is the Rose,
The Moon doth with delight
Look round her when the heavens are bare;
Waters on a starry night
Are beautiful and fair;
The sunshine is a glorious birth;
But yet I know, where'er I go,
That there hath past away a glory from the earth.


III

Now, while the birds thus sing a joyous song,
And while the young lambs bound
As to the tabor's sound,
To me alone there came a thought of grief:
A timely utterance gave that thought relief,
And I again am strong:
The cataracts blow their trumpets from the steep;
No more shall grief of mine the season wrong;
I hear the Echoes through the mountains throng,
The Winds come to me from the fields of sleep,
And all the earth is gay;
Land and sea
Give themselves up to jollity,
And with the heart of May
Doth every Beast keep holiday; -
Thou Child of Joy,
Shout round me, let me hear thy shouts, thou happy Shepherd-boy!


IV

Ye blesse`d Creatures, I have heard the call
Ye to each other make; I see
The heavens laugh with you in your jubilee;
My heart is at your festival,
My head hath its coronal,
The fulness of your bliss, I feel- I feel it all.
Oh evil day! if I were sullen
While the Earth herself is adorning,
This sweet May-morning,
And the Children are culling
On every side,
In a thousand valleys far and wide,
Fresh flowers; while the sun shines warm,
And the Babe leaps up on his Mother's arm:-
I hear, I hear, with joy I hear!
- But there's a Tree, of many, one,
A single Field which I have looked upon,
Both of them speak of something that is gone:
The Pansy at my feet
Doth the same tale repeat:
Whither is fled the visionary gleam?
Where is it now, the glory and the dream?


V

Our birth is but a sleep and a forgetting:
The Soul that rises with us, our life's Star,
Hath had elsewhere its setting,
And cometh from afar:
Not in entire forgetfulness,
And not in utter nakedness,
But trailing clouds of glory do we come
From God, who is our home:
Heaven lies about us in our infancy!
Shades of the prison-house begin to close
Upon the growing Boy,
But He beholds the light, and whence it flows,
He sees it in his joy;
The Youth, who daily farther from the east
Must travel, still is Nature's Priest,
And by the vision splendid
Is on his way attended;
At length the Man perceives it die away,
And fade into the light of common day.


VI

Earth fills her lap with pleasures of her own;
Yearnings she hath in her own natural kind,
And, even with something of a Mother's mind,
And no unworthy aim,
The homely Nurse doth all she can
To make her Foster-child, her Inmate Man,
Forget the glories he hath known,
And that imperial palace whence he came.


VII

Behold the Child among his new-born blisses,
A six years' Darling of a pigmy size!
See, where 'mid work of his own hand he lies,
Fretted by sallies of his mother's kisses,
With light upon him from his father's eyes!
See, at his feet, some little plan or chart,
Some fragment from his dream of human life,
Shaped by himself with newly-learned art;
A wedding or a festival,
A mourning or a funeral;
And this hath now his heart,
And unto this he frames his song:
Then will he fit his tongue
To dialogues of business, love, or strife;
But it will not be long
Ere this be thrown aside,
And with new joy and pride
The little Actor cons another part;
Filling from time to time his 'humorous stage'
With all the Persons, down to palsied Age,
That Life brings with her in her equipage;
As if his whole vocation
Were endless imitation.


VIII

Thou, whose exterior semblance doth belie
Thy Soul's immensity;
Thou best Philosopher, who yet dost keep
Thy heritage, thou Eye among the blind,
That, deaf and silent, read'st the eternal deep,
Haunted for ever by the eternal mind,-
Might Prophet! Seer blest!
On whom those truths do rest,
Which we are toiling all our lives to find,
In darkness lost, the darkness of the grave;
Thou, over whom thy Immortality
Broods like the Day, a Master o'er a Slave,
A Presence which is not to be put by;
[To whom the grave
Is but a lonely bed without the sense or sight
Of day or the warm light,
A place of thought where we in waiting lie; ]
Thou little Child, yet glorious in the might
Of heaven-born freedom on thy being's height,
Why with such earnest pains dost thou provoke
The years to bring the inevitable yoke,
Thus blindly with thy blessedness at strife?
Full soon thy Soul shall have her earthly freight,
And custom lie upon thee with a weight,
Heavy as frost, and deep almost as life!


IX

O joy! that in our embers
Is something that doth live,
That nature yet remembers
What was so fugitive!
The thought of our past years in me doth breed
Perpetual benediction: not indeed
For that which is most worthy to be blest;
Delight and liberty, the simple creed
Of Childhood, whether busy or at rest,
With new-fledged hope still fluttering in his breast:-
Not for these I raise
The song of thanks and praise;
But for those obstinate questionings
Of sense and outward things,
Fallings from us, vanishings;
Blank misgivings of a Creature
Moving about in worlds not realised,
High instincts before which our mortal Nature
Did tremble like a guilty Thing surprised:
But for those first affections,
Those shadowy recollections,
Which, be they what they may,
Are yet the fountain-light of all our day,
Are yet a master-light of all our seeing;
Uphold us, cherish, and have power to make
Our noisy years seem moments in the being
Of the eternal Silence: truths that wake,
To perish never;
Which neither listlessness, nor mad endeavor,
Nor Man nor Boy,
Nor all that is at enmity with joy,
Can utterly abolish or destroy!
Hence in a season of calm weather
Though inland far we be,
Our Souls have sight of that immortal sea
Which brought us hither,
Can in a moment travel thither,
And see the Children sport upon the shore,
And hear the mighty waters rolling evermore.


X

Then sing, ye Birds, sing, sing a joyous song!
And yet the young Lambs bound
As to the tabor's sound!
We in thought will join your throng,
Ye that pipe and ye that play,
Ye that through your hearts to-day
Feel the gladness of the May!
What though the radiance which was once so bright
Be now for ever taken from my sight,
Though nothing can bring back the hour
Of splendour in the grass, of glory in the flower;
We will grieve not, rather find
Strength in what remains behind;
In the primal sympathy
Which having been must ever be;
In the soothing thoughts that spring
Out of human suffering;
In the faith that looks through death,
In years that bring the philosophic mind.


XI

And O, ye Fountains, Meadows, Hills, and Groves,
Forebode not any severing of our loves!
Yet in my heart of hearts I feel your might;
I only have relinquished one delight
To live beneath your more habitual sway.
I love the Brooks which down their channels fret,
Even more than when I tripped lightly as they;
The innocent brightness of a new-born Day
Is lovely yet;
The Clouds that gather round the setting sun
Do take a sober colouring from an eye
That hath kept watch o'er man's mortality;
Another race hath been, and other palms are won.
Thanks to the human heart by which we live,
Thanks to its tenderness, its joys, and fears,
To me the meanest flower that blows can give
Thoughts that do often lie too deep for tears.

Art thou a Statist in the van
Of public conflicts trained and bred?
- First learn to love one living man;
'Then' may'st thou think upon the dead.

A Lawyer art thou? - draw not nigh!
Go, carry to some fitter place
The keenness of that practised eye,
The hardness of that sallow face.

Art thou a Man of purple cheer?
A rosy Man, right plump to see?
Approach; yet, Doctor, not too near,
This grave no cushion is for thee.

Or art thou one of gallant pride,
A Soldier and no man of chaff?
Welcome! - but lay thy sword aside,
And lean upon a peasant's staff.

Physician art thou? one, all eyes,
Philosopher! a fingering slave,
One that would peep and botanise
Upon his mother's grave?

Wrapt closely in thy sensual fleece,
O turn aside,- and take, I pray,
That he below may rest in peace,
Thy ever-dwindling soul, away!

A Moralist perchance appears;
Led, Heaven knows how! to this poor sod:
And he has neither eyes nor ears;
Himself his world, and his own God;

One to whose smooth-rubbed soul can cling
Nor form, nor feeling, great or small;
A reasoning, self-sufficing thing,
An intellectual All-in-all!

Shut close the door; press down the latch;
Sleep in thy intellectual crust;
Nor lose ten tickings of thy watch
Near this unprofitable dust.

But who is He, with modest looks,
And clad in homely russet brown?
He murmurs near the running brooks
A music sweeter than their own.

He is retired as noontide dew,
Or fountain in a noon-day grove;
And you must love him, ere to you
He will seem worthy of your love.

The outward shows of sky and earth,
Of hill and valley, he has viewed;
And impulses of deeper birth
Have come to him in solitude.

In common things that round us lie
Some random truths he can impart,-
The harvest of a quiet eye
That broods and sleeps on his own heart.

But he is weak; both Man and Boy,
Hath been an idler in the land;
Contented if he might enjoy
The things which others understand.

- Come hither in thy hour of strength;
Come, weak as is a breaking wave!
Here stretch thy body at full length;
Or build thy house upon this grave.

--------A Simple Child,
That lightly draws its breath,
And feels its life in every limb,
What should it know of death?

I met a little cottage Girl:
She was eight years old, she said;
Her hair was thick with many a curl
That clustered round her head.

She had a rustic, woodland air,
And she was wildly clad:
Her eyes were fair, and very fair;
--Her beauty made me glad.

"Sisters and brothers, little Maid,
How many may you be?"
"How many? Seven in all," she said
And wondering looked at me.

"And where are they? I pray you tell."
She answered, "Seven are we;
And two of us at Conway dwell,
And two are gone to sea.

"Two of us in the church-yard lie,
My sister and my brother;
And, in the church-yard cottage, I
Dwell near them with my mother."

"You say that two at Conway dwell,
And two are gone to sea,
Yet ye are seven!--I pray you tell,
Sweet Maid, how this may be."

Then did the little Maid reply,
"Seven boys and girls are we;
Two of us in the church-yard lie,
Beneath the church-yard tree."

"You run about, my little Maid,
Your limbs they are alive;
If two are in the church-yard laid,
Then ye are only five."

"Their graves are green, they may be seen,"
The little Maid replied,
"Twelve steps or more from my mother's door,
And they are side by side.

"My stockings there I often knit,
My kerchief there I hem;
And there upon the ground I sit,
And sing a song to them.

"And often after sunset, Sir,
When it is light and fair,
I take my little porringer,
And eat my supper there.

"The first that died was sister Jane;
In bed she moaning lay,
Till God released her of her pain;
And then she went away.

"So in the church-yard she was laid;
And, when the grass was dry,
Together round her grave we played,
My brother John and I.

"And when the ground was white with snow,
And I could run and slide,
My brother John was forced to go,
And he lies by her side."

"How many are you, then," said I,
"If they two are in heaven?"
Quick was the little Maid's reply,
"O Master! we are seven."

"But they are dead; those two are dead!
Their spirits are in heaven!"
'Twas throwing words away; for still
The little Maid would have her will,
And said, "Nay, we are seven!" '''

In [31]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [32]:
tokens = word_tokenize(document.lower())

In [33]:
vocab = {'<unk>': 0}
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

In [34]:
len(vocab)

3067

In [35]:
input_sentences = document.split('\n')

In [36]:
def text_to_indices(sentence, vocab):
  numerical_sentence = []
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])
  return numerical_sentence

In [37]:
input_numerical_sentences = []
for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

In [38]:
len(input_numerical_sentences)

2086

In [39]:
training_sequence = []
for sentence in input_numerical_sentences:
  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [40]:
len_list = []
for sequence in training_sequence:
  len_list.append(len(sequence))
max(len_list)

70

In [41]:
padded_training_sequence = []
for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(len_list)-len(sequence)) + sequence)

In [42]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [43]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:, -1]

## Dataset and DataLoader

In [44]:
class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, index):
    return self.X[index], self.y[index]

In [45]:
dataset = CustomDataset(X, y)

In [46]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## Model

In [47]:
class LSTMModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.linear = nn.Linear(150, vocab_size)

  def forward(self, x):
    x = self.embedding(x)
    _, (x,_) = self.lstm(x)
    x = self.linear(x.squeeze(0))
    return x

In [48]:
model = LSTMModel(len(vocab))

In [49]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

LSTMModel(
  (embedding): Embedding(3067, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (linear): Linear(in_features=150, out_features=3067, bias=True)
)

In [50]:
epochs = 50
learning_rate = 0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [51]:
for epoch in range(epochs):
  total_loss = 0
  for batch_x, batch_y in dataloader:
    batch_x = batch_x.to(device)
    batch_y = batch_y.to(device)
    optimizer.zero_grad()
    output = model(batch_x)
    loss = criterion(output, batch_y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f'Epoch: {epoch+1}, Loss: {total_loss/len(dataloader)}')

Epoch: 1, Loss: 6.43074006422431
Epoch: 2, Loss: 5.251686211708373
Epoch: 3, Loss: 4.175760012284845
Epoch: 4, Loss: 3.3476228476625627
Epoch: 5, Loss: 2.7362951609940653
Epoch: 6, Loss: 2.313370035021706
Epoch: 7, Loss: 1.995938759593837
Epoch: 8, Loss: 1.7255497740960755
Epoch: 9, Loss: 1.5208428011531323
Epoch: 10, Loss: 1.3907382890450215
Epoch: 11, Loss: 1.287041632475051
Epoch: 12, Loss: 1.250196016863384
Epoch: 13, Loss: 1.18959493552689
Epoch: 14, Loss: 1.2007776014836489
Epoch: 15, Loss: 1.2002562335239046
Epoch: 16, Loss: 1.1377733335294555
Epoch: 17, Loss: 1.0379086375500248
Epoch: 18, Loss: 1.00509241401358
Epoch: 19, Loss: 1.0072197205329363
Epoch: 20, Loss: 1.0301739447544107
Epoch: 21, Loss: 1.0822926917962268
Epoch: 22, Loss: 1.057563789056993
Epoch: 23, Loss: 1.0431413266115483
Epoch: 24, Loss: 1.0058480780077192
Epoch: 25, Loss: 1.0062396869195247
Epoch: 26, Loss: 1.0520533494717252
Epoch: 27, Loss: 1.0078845730925028
Epoch: 28, Loss: 0.9683107782847586
Epoch: 29, Los

In [52]:
def prediction(model, vocab, text):
  tokenized_text = word_tokenize(text.lower())
  numerical_text = text_to_indices(tokenized_text, vocab)
  padded_text = torch.tensor([0]*(max(len_list)-len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0).to(device)
  output = model(padded_text)
  value, index = torch.max(output, dim=1)
  return text + " " + list(vocab.keys())[index]

In [54]:
num_tokens = 10
input_text = "Birds fly"
for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text

Birds fly and
Birds fly and upright
Birds fly and upright figure
Birds fly and upright figure in
Birds fly and upright figure in a
Birds fly and upright figure in a strange
Birds fly and upright figure in a strange world
Birds fly and upright figure in a strange world to
Birds fly and upright figure in a strange world to all
Birds fly and upright figure in a strange world to all .
